# SST Decision Navigator

### A Semantic Search Tool for Social Security Tribunal of Canada Decisions

**Course:** Legal Information Technology

## Introduction

Self-represented litigants before Canada's Social Security Tribunal (SST) face a
difficult challenge: finding past decisions relevant to their situation among
thousands of published rulings. Traditional keyword search fails because legal
concepts can be expressed in many different ways — a claimant searching for
"fired for being late" needs to find decisions about "misconduct" and
"just cause."

This tool addresses that gap with a **two-stage Retrieval-Augmented Generation
(RAG) pipeline**:

1. **Stage 1 — Semantic Search (Bi-Encoder):** Embeds both the user's query and
   all 17,000+ SST decisions into a shared vector space using
   [Qwen3-Embedding-8B](https://huggingface.co/Qwen/Qwen3-Embedding-8B), then
   retrieves the top 40 candidates by cosine similarity.

2. **Stage 2 — Cross-Encoder Reranking:** Scores each candidate with a
   cross-encoder model
   ([Qwen3-Reranker-8B](https://huggingface.co/Qwen/Qwen3-Reranker-8B)) that
   reads the query and document *together*, using a **section-aware packing**
   strategy to prioritize the most legally salient parts of each decision.

3. **Stage 3 — Case Card Generation:** Produces a plain-language four-section
   summary of the top result using
   [Qwen3-14B](https://huggingface.co/Qwen/Qwen3-14B), designed to be
   understandable by non-lawyers.

### Architecture

```
User Query
    │
    ▼
┌─────────────────────────────────┐
│  Stage 1: Bi-Encoder            │  Qwen3-Embedding-8B
│  Semantic Search                │  Cosine similarity → Top 40 candidates
└───────────────┬─────────────────┘
                │
                ▼
┌─────────────────────────────────┐
│  Stage 2: Cross-Encoder         │  Qwen3-Reranker-8B
│  Reranking                      │  Section-aware packing → Top 5 results
└───────────────┬─────────────────┘
                │
                ▼
┌─────────────────────────────────┐
│  Stage 3: Case Card Generation  │  Qwen3-14B
│  Plain-language summary         │  Issue / Key Facts / Test / Outcome
└─────────────────────────────────┘
```

### Data Source

Decisions are sourced from the
[A2AJ Canadian Case Law](https://huggingface.co/datasets/a2aj/canadian-case-law)
dataset on Hugging Face, which provides the full text of published SST rulings.
Pre-computed embeddings for the entire corpus are cached on Hugging Face to avoid
the expensive one-time encoding step.

---
## 1. Setup

Install the required dependencies and configure the API key for cloud inference.

In [ ]:
%pip install -q pandas pyarrow numpy huggingface_hub openai requests certifi

In [ ]:
import logging
import json
import os
import re
import ssl
from dataclasses import dataclass, field
from getpass import getpass
from pathlib import Path
from typing import NamedTuple

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    import certifi
except ImportError:
    certifi = None

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)-30s  %(levelname)-8s  %(message)s",
)
logger = logging.getLogger("sst_navigator")

# ── DeepInfra API key ────────────────────────────────────────────────
# Required for Stage 1 query embedding, Stage 2 reranking, and
# Stage 3 generation.  Free tier available at https://deepinfra.com
if not os.environ.get("DEEPINFRA_API_KEY"):
    os.environ["DEEPINFRA_API_KEY"] = getpass("Enter your DeepInfra API key: ")

print("Setup complete.")

---
## 2. Configuration

All model identifiers, pipeline parameters, and API endpoints are defined here.
These constants are referenced throughout the pipeline.

In [ ]:
# ── Model identifiers (DeepInfra cloud) ──────────────────────────────
DEEPINFRA_BASE_URL         = "https://api.deepinfra.com/v1/openai"
DEEPINFRA_EMBEDDING_MODEL  = "Qwen/Qwen3-Embedding-8B"
DEEPINFRA_RERANKER_MODEL   = "Qwen/Qwen3-Reranker-8B"
DEEPINFRA_RERANKER_ENDPOINT = (
    "https://api.deepinfra.com/v1/inference/Qwen/Qwen3-Reranker-8B"
)
DEEPINFRA_GENERATION_MODEL = "Qwen/Qwen3-14B"

# ── Pipeline parameters ─────────────────────────────────────────────
STAGE1_TOP_K   = 40     # Candidates from semantic search
STAGE2_TOP_K   = 5      # Final results after cross-encoder reranking
SNIPPET_LENGTH = 500    # Characters for the preview snippet

# Fast-mode reduces latency at a small quality cost
FAST_STAGE1_TOP_K = 20
FAST_STAGE2_TOP_K = 3

# ── Embedding parameters ────────────────────────────────────────────
EMBEDDING_INSTRUCTION = (
    "Represent this query for retrieving similar Canadian "
    "administrative legal tribunal decisions"
)
EMBEDDING_MAX_TOKENS       = 8192
FAST_EMBEDDING_MAX_TOKENS  = 4096

# Pre-computed embedding cache hosted on HuggingFace
EMBEDDING_CACHE_DIR        = ".cache/embeddings"
EMBEDDING_CACHE_REPO_ID    = "mystic63/sst-embeddings-cache"
EMBEDDING_CACHE_REPO_TYPE  = "dataset"
EMBEDDING_CACHE_FILE       = "sst_embeddings_qwen3_8b.npy"
EMBEDDING_METADATA_FILE    = "metadata.json"

# ── Reranker parameters ─────────────────────────────────────────────
RERANKER_INSTRUCTION = (
    "Given a legal query describing an appellant's situation, "
    "retrieve relevant Social Security Tribunal of Canada decisions"
)
RERANKER_MAX_TOKENS      = 8192
FAST_RERANKER_MAX_TOKENS = 2048

# ── Generation parameters ───────────────────────────────────────────
GENERATION_MAX_TOKENS      = 512
FAST_GENERATION_MAX_TOKENS = 256
GENERATION_MAX_CHARS       = 24_000
FAST_GENERATION_MAX_CHARS  = 12_000

# ── Data ─────────────────────────────────────────────────────────────
DEV_ROW_LIMIT = 500   # Row limit for fast development iteration

print("Configuration loaded.")

---
## 3. Data Loading

SST decisions are loaded from the
[A2AJ Canadian Case Law](https://huggingface.co/datasets/a2aj/canadian-case-law)
dataset on Hugging Face. The dataset is distributed as a Parquet file and
contains the full text of every published SST ruling along with metadata
(case name, date, URL).

Key cleaning steps:
- Drop rows where the decision text is empty or null
- Verify that the expected columns exist
- Optionally trim to a subset for faster development

In [ ]:
PARQUET_URL = (
    "https://huggingface.co/datasets/a2aj/canadian-case-law"
    "/resolve/main/SST/train.parquet"
)
REQUIRED_COLUMNS = ["name_en", "document_date_en", "url_en", "unofficial_text_en"]


def _is_ssl_cert_error(error: Exception) -> bool:
    """Check if an exception is caused by an SSL certificate error."""
    return "CERTIFICATE_VERIFY_FAILED" in str(error)


def _configure_certifi_ca_bundle() -> bool:
    """Use certifi's CA bundle as a fallback for HTTPS connections."""
    if certifi is None:
        return False
    ca_path = certifi.where()
    os.environ.setdefault("SSL_CERT_FILE", ca_path)
    os.environ.setdefault("REQUESTS_CA_BUNDLE", ca_path)
    ssl._create_default_https_context = lambda: ssl.create_default_context(
        cafile=ca_path
    )
    return True


def load_sst_decisions(max_rows: int | None = None) -> pd.DataFrame:
    """Load SST decisions from the A2AJ Hugging Face dataset.

    Args:
        max_rows: If set, only load the first N rows (useful for dev/testing).

    Returns:
        DataFrame with cleaned SST decisions.
    """
    logger.info(
        "Loading SST decisions from Hugging Face (%s rows)...",
        max_rows or "all",
    )
    try:
        df = pd.read_parquet(PARQUET_URL)
    except Exception as e:
        if _is_ssl_cert_error(e) and _configure_certifi_ca_bundle():
            logger.warning(
                "SSL certificate verification failed. "
                "Retrying with certifi CA bundle."
            )
            df = pd.read_parquet(PARQUET_URL)
        else:
            raise RuntimeError(
                "Could not load the SST dataset. "
                f"Check your internet connection.\n{e}"
            ) from e

    # Verify expected columns exist
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Dataset is missing expected columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    # Drop rows where the decision text is empty or null
    df = df.dropna(subset=["unofficial_text_en"])
    df = df[df["unofficial_text_en"].str.strip().astype(bool)]
    df = df.reset_index(drop=True)

    if max_rows is not None:
        df = df.head(max_rows)
        logger.info("Trimmed to %d rows for development.", len(df))

    logger.info("Loaded %d SST decisions.", len(df))
    return df

In [ ]:
# Load the full dataset (set max_rows=DEV_ROW_LIMIT for faster iteration)
df = load_sst_decisions(max_rows=None)

print(f"Total decisions: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['document_date_en'].min()} to {df['document_date_en'].max()}")
print(f"Average text length: {df['unofficial_text_en'].str.len().mean():,.0f} characters")
df[["name_en", "document_date_en", "url_en"]].head(10)

---
## 4. Section-Aware Document Parsing

SST decisions follow a standard structure with recognisable headings:
**Overview → Issue → The Law → Analysis → Conclusion**. Not every decision uses
the same headings, but the pattern is consistent enough to exploit.

Our section parser:
1. **Strips boilerplate** — removes metadata headers, table-of-contents lines,
   and trailing footnote blocks.
2. **Detects section headings** — uses a regex to find capitalised headings
   followed by paragraph markers like `[1]`, `[2]`, etc.
3. **Assigns a priority** to each section based on its heading, ranking the
   most legally salient content highest:

| Priority | Headings | Why |
|----------|----------|-----|
| 1 (highest) | Analysis, Law and Analysis | Contains the legal test and the tribunal's reasoning |
| 2 | Issue, What I have to decide | Frames the legal question |
| 3 | Conclusion, Decision | States the outcome |
| 4 | Overview, Introduction, Background | Context |
| 5 | The Law, Applicable Law | Statutory provisions (usually boilerplate) |
| 6 | Evidence, Submissions | Factual record |
| 7 | Everything else | Custom or unusual headings |

4. **Greedily packs** sections into a character budget for the reranker,
   starting with the highest-priority sections. Analysis sections receive
   special truncation: the head (which states the legal test) and the last
   two paragraphs (which contain the finding) are kept.

This approach ensures the cross-encoder sees the most decision-relevant text
even when the full document far exceeds the model's context window.

In [ ]:
# ── Section data structure ───────────────────────────────────────────

class Section(NamedTuple):
    name: str       # heading text, e.g. "Analysis"
    text: str       # body text (heading + paragraphs)
    priority: int   # 1 = highest value for relevance scoring
    order: int      # original position in the document


# ── Priority mapping ─────────────────────────────────────────────────
# Ordered list of (compiled pattern, priority).  First match wins.

_PRIORITY_RULES: list[tuple[re.Pattern, int]] = [
    (re.compile(r"analysis|law and analysis", re.IGNORECASE), 1),
    (re.compile(
        r"issue|issues|what i have to decide|what i must decide"
        r"|what the .* must prove",
        re.IGNORECASE,
    ), 2),
    (re.compile(r"conclusion|decision", re.IGNORECASE), 3),
    (re.compile(r"overview|introduction|background", re.IGNORECASE), 4),
    (re.compile(
        r"the law|applicable law|what the law says|legal framework",
        re.IGNORECASE,
    ), 5),
    (re.compile(
        r"evidence|submissions|what the .* says",
        re.IGNORECASE,
    ), 6),
]

_DEFAULT_PRIORITY = 7


def _heading_priority(name: str) -> int:
    """Return the priority tier (1 = highest) for a section heading."""
    for pattern, priority in _PRIORITY_RULES:
        if pattern.search(name):
            return priority
    return _DEFAULT_PRIORITY


# ── Boilerplate stripping ────────────────────────────────────────────

def _strip_boilerplate(text: str) -> str:
    """Remove metadata header and trailing footnote block."""
    # Strip metadata before "Decision Content"
    dc_idx = text.find("Decision Content")
    if dc_idx != -1:
        text = text[dc_idx + len("Decision Content"):]

    # Skip "On this page" TOC line
    toc_idx = text.find("On this page")
    if toc_idx != -1:
        after_toc = text.find("\n", toc_idx)
        if after_toc != -1:
            next_newline = text.find("\n", after_toc + 1)
            if next_newline != -1:
                text = text[next_newline + 1:]
            else:
                text = text[after_toc + 1:]
        else:
            text = text[toc_idx + len("On this page"):]

    # Skip "Reasons and decision" bridge line (older decisions)
    rd_match = re.match(r"\s*Reasons and decision\s*", text)
    if rd_match:
        text = text[rd_match.end():]

    # Strip trailing footnote block
    fn_match = re.search(r"\nFootnote 1\n", text)
    if fn_match:
        text = text[:fn_match.start()]

    return text.strip()


# ── Section splitting ────────────────────────────────────────────────
# Matches a heading followed (possibly with whitespace) by a [N]
# paragraph marker.  The heading starts with a capital letter and
# contains letters, spaces, hyphens, dashes, commas, apostrophes,
# and slashes.

_HEADING_RE = re.compile(
    r"^([A-Z][A-Za-z][A-Za-z \-\u2013\u2014,/\']{0,80}?)"
    r"\s*"
    r"(?=\[\d+\])",
    re.MULTILINE,
)


def parse_sections(text: str) -> list[Section]:
    """Parse an SST decision into prioritised sections.

    Returns a list of Section objects sorted by priority (ascending)
    with document order as tiebreaker within the same tier.
    """
    cleaned = _strip_boilerplate(text)
    if not cleaned:
        return []

    matches = list(_HEADING_RE.finditer(cleaned))
    if not matches:
        # No headings found — return the entire text as one section.
        return [Section(
            name="body", text=cleaned,
            priority=_DEFAULT_PRIORITY, order=0,
        )]

    sections: list[Section] = []
    for i, m in enumerate(matches):
        name = m.group(1).strip()
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(cleaned)
        body = cleaned[start:end].strip()
        sections.append(Section(
            name=name,
            text=body,
            priority=_heading_priority(name),
            order=i,
        ))

    return sections


# ── Greedy packing ───────────────────────────────────────────────────

_PARAGRAPH_RE = re.compile(r"\[\d+\]")
_SECTION_SEP = "\n\n"


def _truncate_analysis(text: str, budget: int) -> str:
    """Truncate an Analysis section keeping the head and last 2 paragraphs.

    The head typically contains the legal test the tribunal applies,
    while the final paragraphs contain the finding.  Keeping both gives
    the cross-encoder the most decision-relevant signal.
    """
    if len(text) <= budget:
        return text

    tail_budget = budget // 5
    head_budget = budget - tail_budget - len(" [...] ")

    para_matches = list(_PARAGRAPH_RE.finditer(text))
    if len(para_matches) >= 3:
        tail_start = para_matches[-2].start()
        tail = text[tail_start:]
        if len(tail) > tail_budget:
            tail = tail[:tail_budget]
    else:
        tail = ""
        head_budget = budget

    head = text[:head_budget]
    if tail:
        return head + " [...] " + tail
    return head


def pack_for_reranker(text: str, char_budget: int) -> str:
    """Pack an SST decision into a character budget for the reranker.

    Extracts sections, prioritises legally salient content (Analysis,
    Issue, Conclusion), and greedily fills the budget.  Falls back to
    head-truncation when section parsing fails.
    """
    sections = parse_sections(text)
    if not sections:
        return text[:char_budget]

    # If the entire cleaned text fits, just return it in document order.
    total = (
        sum(len(s.text) for s in sections)
        + len(_SECTION_SEP) * (len(sections) - 1)
    )
    if total <= char_budget:
        return _SECTION_SEP.join(
            f"[{s.name}] {s.text}"
            for s in sorted(sections, key=lambda s: s.order)
        )

    # Sort by (priority, original order) — highest-value first.
    ordered = sorted(sections, key=lambda s: (s.priority, s.order))

    packed: list[tuple[int, str]] = []  # (original order, formatted text)
    remaining = char_budget

    for s in ordered:
        if remaining <= 0:
            break

        label = f"[{s.name}] "
        label_len = len(label)
        body_budget = remaining - label_len - len(_SECTION_SEP)
        if body_budget <= 0:
            break

        if len(s.text) <= body_budget:
            packed.append((s.order, label + s.text))
            remaining -= len(label) + len(s.text) + len(_SECTION_SEP)
        else:
            if s.priority == 1:
                truncated = _truncate_analysis(s.text, body_budget)
            else:
                truncated = s.text[:body_budget]
            packed.append((s.order, label + truncated))
            remaining -= len(label) + len(truncated) + len(_SECTION_SEP)

    # Reassemble in original document order.
    packed.sort(key=lambda t: t[0])
    return _SECTION_SEP.join(text for _, text in packed)


print("Section parser defined.")

In [ ]:
# Demo: parse a real SST decision into sections
sample_decision = df.iloc[0]["unofficial_text_en"]
sections = parse_sections(sample_decision)

print(f"Decision: {df.iloc[0]['name_en']}")
print(f"Sections found: {len(sections)}\n")
print(f"{'Section':<40} {'Priority':>8}  {'Length':>8}")
print("-" * 60)
for s in sorted(sections, key=lambda s: (s.priority, s.order)):
    print(f"{s.name:<40} {s.priority:>8}  {len(s.text):>8,}")

# Show how packing works with a small budget
packed = pack_for_reranker(sample_decision, char_budget=4000)
print(f"\n--- Packed for reranker (budget=4000 chars, used={len(packed):,}) ---")
print(packed[:1500] + "\n[...]" if len(packed) > 1500 else packed)

---
## 5. Stage 1 — Semantic Search (Bi-Encoder)

The first stage uses a **bi-encoder** architecture: the query and every document
are independently embedded into the same vector space, and candidates are
retrieved by cosine similarity.

### How it works

1. **Pre-computed document embeddings** are loaded from a cache on Hugging Face.
   Encoding 17,000+ decisions is expensive (~hours on a GPU), so we do it once
   and cache the results as a numpy `.npy` file alongside a `metadata.json`
   that records the URL ordering.

2. **At query time**, only the short user query is embedded via the DeepInfra
   API (Qwen3-Embedding-8B), wrapped with a task-specific instruction:
   > *"Represent this query for retrieving similar Canadian administrative
   > legal tribunal decisions"*

3. **Cosine similarity** between the query vector and all document vectors
   is computed as a single matrix multiplication (since all vectors are
   unit-normalised, dot product = cosine similarity). The top 40 candidates
   are returned.

### Embedding quality safeguards

- **Sanitisation:** Any NaN/Inf values in embedding vectors (rare but possible
  with quantised models) are replaced with zeros.
- **L2 normalisation:** Performed in float64 to avoid overflow when squaring
  large float32 values, then downcast back to float32 for efficient dot product.
- **Alignment:** The cached vectors may be in a different order than the
  DataFrame rows (the cache-building script sorts by text length for efficient
  batching). The `align_to_urls` method reorders vectors to match the DataFrame.

In [ ]:
# ── Embedding utility functions ──────────────────────────────────────

def _sanitize_embedding_batch(result: np.ndarray) -> None:
    """Zero out rows containing NaN/Inf in *result* in-place.

    Rows replaced with zeros will produce a cosine similarity of 0
    and rank last in search results.
    """
    bad_rows = ~np.isfinite(result).all(axis=1)
    if bad_rows.any():
        n_bad = int(bad_rows.sum())
        logger.warning(
            "%d of %d embedding vectors contain NaN/Inf — "
            "replacing with zeros.",
            n_bad, result.shape[0],
        )
        result[bad_rows] = 0.0


def _l2_normalize_rows(arr: np.ndarray) -> np.ndarray:
    """Return a copy of *arr* with every row L2-normalised.

    Zero-norm rows (e.g. previously sanitised bad vectors) are left
    as zeros rather than producing NaN from 0/0.

    The norm is computed in float64 to avoid overflow when squaring
    large float32 values (e.g. 1e20² exceeds float32 max).
    """
    arr64 = arr.astype(np.float64)
    norms = np.linalg.norm(arr64, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return (arr64 / norms).astype(np.float32)


def _sanitize_and_normalize_rows(arr: np.ndarray) -> np.ndarray:
    """Return a sanitized + unit-normalized copy of *arr*."""
    clean = np.nan_to_num(arr, copy=True, nan=0.0, posinf=0.0, neginf=0.0)
    return _l2_normalize_rows(clean)


# ── SemanticSearcher class ───────────────────────────────────────────

class SemanticSearcher:
    """Embeds queries and searches a pre-computed document embedding index.

    Uses the DeepInfra API (Qwen3-Embedding-8B) for query encoding and
    pre-computed cached vectors for the document corpus.
    """

    def __init__(self):
        self._doc_embeddings: np.ndarray | None = None
        self._cache_urls: list[str] | None = None

    # -- Persistent cache ------------------------------------------------

    def load_embeddings_cache(
        self,
        cache_dir: str = EMBEDDING_CACHE_DIR,
        repo_id: str = EMBEDDING_CACHE_REPO_ID,
        repo_type: str = EMBEDDING_CACHE_REPO_TYPE,
        embeddings_file: str = EMBEDDING_CACHE_FILE,
        metadata_file: str = EMBEDDING_METADATA_FILE,
    ) -> bool:
        """Download and load cached embeddings from HuggingFace.

        The embedding vectors (.npy) and metadata (URL ordering, .json)
        are downloaded if not already present locally.
        """
        base = Path(cache_dir)
        base.mkdir(parents=True, exist_ok=True)
        emb_path = base / embeddings_file
        meta_path = base / metadata_file

        # Download from HuggingFace if missing locally
        if not emb_path.exists() or not meta_path.exists():
            try:
                hf_hub_download(
                    repo_id=repo_id,
                    filename=embeddings_file,
                    repo_type=repo_type,
                    local_dir=str(base),
                    local_dir_use_symlinks=False,
                )
                hf_hub_download(
                    repo_id=repo_id,
                    filename=metadata_file,
                    repo_type=repo_type,
                    local_dir=str(base),
                    local_dir_use_symlinks=False,
                )
                logger.info("Downloaded embedding cache from %s", repo_id)
            except Exception as exc:
                logger.warning(
                    "Could not download embedding cache: %s", exc
                )

        if not emb_path.exists() or not meta_path.exists():
            return False

        # Parse metadata to get the URL ordering
        try:
            raw = json.loads(meta_path.read_text(encoding="utf-8"))
        except Exception as exc:
            logger.error("Invalid metadata JSON: %s", exc)
            return False

        if isinstance(raw, list):
            self._cache_urls = [str(u) for u in raw]
        elif isinstance(raw, dict) and "url_en" in raw:
            self._cache_urls = [str(u) for u in raw["url_en"]]
        else:
            logger.error("Unexpected metadata format.")
            return False

        self._doc_embeddings = np.load(
            emb_path, allow_pickle=False,
        ).astype(np.float32, copy=False)

        # Validate consistency
        if self._doc_embeddings.shape[0] != len(self._cache_urls):
            logger.error(
                "Mismatch: %d vectors vs %d URLs.",
                self._doc_embeddings.shape[0], len(self._cache_urls),
            )
            self._doc_embeddings = None
            self._cache_urls = None
            return False

        # Deduplicate URLs (keep first occurrence)
        n_unique = len(set(self._cache_urls))
        if n_unique < len(self._cache_urls):
            seen: set[str] = set()
            keep: list[int] = []
            for i, url in enumerate(self._cache_urls):
                if url not in seen:
                    seen.add(url)
                    keep.append(i)
            self._doc_embeddings = self._doc_embeddings[np.array(keep)]
            self._cache_urls = [self._cache_urls[i] for i in keep]

        logger.info(
            "Loaded embedding cache — %d vectors, dim=%d",
            self._doc_embeddings.shape[0], self._doc_embeddings.shape[1],
        )
        return True

    def align_to_urls(self, target_urls: list[str]) -> list[int]:
        """Reorder cached embeddings to match *target_urls*.

        The cached vectors may be in a different order than the
        DataFrame rows.  This method reorders them so that
        embedding row i corresponds to DataFrame row i.

        Returns the indices of target_urls that had a cache match.
        """
        if self._doc_embeddings is None or self._cache_urls is None:
            raise RuntimeError("Call load_embeddings_cache() first.")

        cache_lookup = {url: i for i, url in enumerate(self._cache_urls)}

        matched_target: list[int] = []
        matched_cache: list[int] = []
        for target_idx, url in enumerate(target_urls):
            cache_idx = cache_lookup.get(url)
            if cache_idx is not None:
                matched_target.append(target_idx)
                matched_cache.append(cache_idx)

        if not matched_cache:
            raise RuntimeError("No cached embeddings match the target URLs.")

        self._doc_embeddings = self._doc_embeddings[np.array(matched_cache)]
        self._cache_urls = None  # no longer needed

        # Sanitise + re-normalise cached vectors
        self._doc_embeddings = _sanitize_and_normalize_rows(
            self._doc_embeddings
        )

        logger.info(
            "Aligned embeddings: %d of %d target URLs matched.",
            len(matched_target), len(target_urls),
        )
        return matched_target

    # -- Query embedding (DeepInfra API) ---------------------------------

    def _embed_query(self, text: str) -> np.ndarray:
        """Embed a single query string via the DeepInfra API."""
        from openai import OpenAI

        api_key = os.environ.get("DEEPINFRA_API_KEY")
        if not api_key:
            raise RuntimeError("Set DEEPINFRA_API_KEY in your environment.")

        client = OpenAI(api_key=api_key, base_url=DEEPINFRA_BASE_URL)
        response = client.embeddings.create(
            input=[text],
            model=DEEPINFRA_EMBEDDING_MODEL,
            encoding_format="float",
        )
        vector = np.array(
            [response.data[0].embedding], dtype=np.float32,
        )
        _sanitize_embedding_batch(vector)
        return vector

    # -- Search ----------------------------------------------------------

    def search(
        self,
        query: str,
        top_k: int = STAGE1_TOP_K,
    ) -> list[tuple[int, float]]:
        """Return the top-K (index, score) pairs for a query.

        The query is wrapped with the retrieval instruction before
        encoding.  Similarity is computed via dot product (vectors
        are already unit-normalised).
        """
        if self._doc_embeddings is None:
            raise RuntimeError("Load embeddings first.")
        if self._doc_embeddings.shape[0] == 0:
            return []

        # Wrap query with the task instruction
        formatted_query = (
            f"Instruct: {EMBEDDING_INSTRUCTION}\n"
            f"Query:{query}"
        )
        q_vec = self._embed_query(formatted_query)
        q_vec = _sanitize_and_normalize_rows(q_vec)

        # Cosine similarity via dot product
        with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
            scores = self._doc_embeddings @ q_vec.T
        scores = scores.squeeze(-1)
        np.nan_to_num(
            scores, copy=False, nan=-1.0, posinf=-1.0, neginf=-1.0,
        )

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

    @property
    def has_embeddings(self) -> bool:
        return (
            self._doc_embeddings is not None
            and self._doc_embeddings.shape[0] > 0
        )


print("SemanticSearcher defined.")

In [ ]:
# Initialize the searcher and load pre-computed embeddings
searcher = SemanticSearcher()

print("Downloading embedding cache from HuggingFace...")
assert searcher.load_embeddings_cache(), "Failed to load embedding cache!"

# Align cached vectors with our DataFrame rows
target_urls = df["url_en"].astype(str).tolist()
matched_indices = searcher.align_to_urls(target_urls)

# Trim DataFrame to only rows with cached embeddings
if len(matched_indices) < len(df):
    df = df.iloc[matched_indices].reset_index(drop=True)
    print(f"Trimmed DataFrame to {len(df):,} rows with cached embeddings.")

print(f"\nEmbedding index ready:")
print(f"  Documents: {searcher._doc_embeddings.shape[0]:,}")
print(f"  Dimensions: {searcher._doc_embeddings.shape[1]}")

In [ ]:
# Run a sample semantic search
sample_query = (
    "I was fired for being late to work multiple times "
    "and my EI benefits were denied for misconduct"
)

print(f"Query: {sample_query!r}\n")
stage1_hits = searcher.search(sample_query, top_k=STAGE1_TOP_K)

print(f"Stage 1 returned {len(stage1_hits)} candidates.\n")
print(f"{'Rank':<6} {'Score':>8}  {'Decision Name'}")
print("-" * 70)
for rank, (idx, score) in enumerate(stage1_hits[:10], 1):
    row = df.iloc[idx]
    name = row["name_en"][:50]
    print(f"{rank:<6} {score:>8.4f}  {name}")
print("...")

---
## 6. Stage 2 — Cross-Encoder Reranking

The bi-encoder from Stage 1 is fast but approximate: it encodes the query
and documents *independently*, so it cannot model fine-grained interactions
between them.

Stage 2 uses a **cross-encoder** (Qwen3-Reranker-8B) that reads the query
and each candidate document *together* and produces a single relevance score.
This is much more accurate but too slow to run over the full corpus — hence the
two-stage approach.

### Section-aware packing

SST decisions can be very long (10,000+ words), but the reranker has a limited
context window. Rather than naively truncating from the beginning, we use the
**section parser** from Section 4 to intelligently pack the most legally
salient sections into the character budget:

1. Parse the decision into sections
2. Sort by priority (Analysis > Issue > Conclusion > ...)
3. Greedily pack sections until the budget is full
4. Reassemble in original document order

This ensures the cross-encoder sees the tribunal's reasoning and conclusion
rather than procedural metadata.

In [ ]:
# ── Reranker system prompt ───────────────────────────────────────────

_RERANKER_SYSTEM_PROMPT = (
    "Judge whether the Document meets the requirements based on the "
    "Query and the Instruct provided. Note that the answer can only "
    'be "yes" or "no".'
)


class Reranker:
    """Cross-encoder reranker using the DeepInfra API.

    Scores query-document pairs using Qwen3-Reranker-8B and returns
    the top-K candidates sorted by relevance.
    """

    def _score_batch(
        self, query: str, documents: list[str],
    ) -> list[float]:
        """Score all documents in one API call via DeepInfra."""
        import requests as req

        api_key = os.environ.get("DEEPINFRA_API_KEY")
        if not api_key:
            raise RuntimeError("Set DEEPINFRA_API_KEY in your environment.")

        # Prepend the task instruction to the query
        formatted_query = f"{RERANKER_INSTRUCTION}\n{query}"

        resp = req.post(
            DEEPINFRA_RERANKER_ENDPOINT,
            headers={
                "Authorization": f"bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={
                "queries": [formatted_query],
                "documents": documents,
            },
            timeout=120,
        )
        resp.raise_for_status()
        data = resp.json()

        scores = data["scores"]
        # API may return [[s1, s2, ...]] (one list per query) or [s1, s2, ...]
        if scores and isinstance(scores[0], list):
            scores = scores[0]
        return [float(s) for s in scores]

    def rerank(
        self,
        query: str,
        candidates: list[dict],
        top_k: int = STAGE2_TOP_K,
        max_tokens: int = RERANKER_MAX_TOKENS,
    ) -> list[dict]:
        """Score and re-sort candidates by cross-encoder relevance.

        Each candidate dict must contain at least 'text' and 'index' keys.
        Returns the top_k candidates sorted by descending reranker score.
        """
        logger.info("Reranking %d candidates...", len(candidates))

        # Pack each document's most salient sections within budget
        documents = []
        for cand in candidates:
            doc_text = pack_for_reranker(
                cand["text"], char_budget=max_tokens * 4,
            )
            documents.append(doc_text)

        # Score all candidates in a single API call
        scores = self._score_batch(query, documents)

        for cand, score in zip(candidates, scores):
            cand["reranker_score"] = score

        ranked = sorted(
            candidates,
            key=lambda c: c["reranker_score"],
            reverse=True,
        )
        return ranked[:top_k]


print("Reranker defined.")

In [ ]:
# Build candidate dicts from Stage 1 hits
candidates = []
for idx, score in stage1_hits:
    row = df.iloc[idx]
    candidates.append({
        "index": idx,
        "name": row.get("name_en", "Unnamed"),
        "date": str(row.get("document_date_en", "")),
        "url": row.get("url_en", ""),
        "text": row["unofficial_text_en"],
        "stage1_score": score,
    })

# Run the cross-encoder reranker
reranker = Reranker()
top_results = reranker.rerank(
    sample_query, candidates,
    top_k=STAGE2_TOP_K,
    max_tokens=RERANKER_MAX_TOKENS,
)

print(f"Stage 2 returned {len(top_results)} results.\n")
print(f"{'Rank':<6} {'Reranker':>10} {'Stage1':>10}  {'Decision Name'}")
print("-" * 75)
for rank, r in enumerate(top_results, 1):
    print(
        f"{rank:<6} {r['reranker_score']:>10.4f} "
        f"{r['stage1_score']:>10.4f}  "
        f"{r['name'][:45]}"
    )

---
## 7. Stage 3 — Case Card Generation

The final stage generates a **plain-language summary** of the top-ranked
decision, designed to help self-represented litigants quickly understand
the key aspects of a case without reading the full text.

The summary follows a fixed four-section format:

- **Issue:** What legal question was the tribunal deciding?
- **Key Facts:** What were the most important facts?
- **Test Applied:** What legal test or criteria did the tribunal use?
- **Outcome:** What did the tribunal decide, and why?

The generation model (Qwen3-14B on DeepInfra) is prompted with a system
message that enforces this structure and instructs the model to use simple
language and not invent facts.

Output is post-processed to remove any `<think>...</think>` reasoning
blocks that the model may produce (Qwen3 models use these for internal
chain-of-thought).

In [ ]:
# ── Case card system prompt ──────────────────────────────────────────

_CASE_CARD_SYSTEM = (
    "You are a legal-aid assistant. Read the tribunal decision below "
    "and produce a plain-language summary a self-represented person "
    "can understand. Output EXACTLY four sections with these "
    "headings:\n\n"
    "**Issue:** (What legal question was the tribunal deciding?)\n"
    "**Key Facts:** (What were the most important facts?)\n"
    "**Test Applied:** (What legal test or criteria did the tribunal "
    "use?)\n"
    "**Outcome:** (What did the tribunal decide, and why?)\n\n"
    "Be concise. Use simple language. Do not invent facts."
)

_THINK_BLOCK_RE = re.compile(
    r"<think>.*?</think>", re.IGNORECASE | re.DOTALL,
)


def _sanitize_case_card_output(text: str) -> str:
    """Remove hidden reasoning blocks, keep the user-facing summary."""
    sanitized = _THINK_BLOCK_RE.sub("", text)

    expected = (
        "**Issue:**", "**Key Facts:**",
        "**Test Applied:**", "**Outcome:**",
    )
    positions = [
        sanitized.find(s) for s in expected if s in sanitized
    ]
    if positions:
        sanitized = sanitized[min(positions):]

    return sanitized.strip()


class CaseCardGenerator:
    """Generate a structured plain-language summary of an SST decision.

    Uses the DeepInfra API (Qwen3-14B) for generation.
    """

    def generate_case_card(
        self,
        decision_text: str,
        max_tokens: int = GENERATION_MAX_TOKENS,
        max_chars: int = GENERATION_MAX_CHARS,
    ) -> str:
        """Return a formatted case-card string for the given decision."""
        from openai import OpenAI

        api_key = os.environ.get("DEEPINFRA_API_KEY")
        if not api_key:
            raise RuntimeError("Set DEEPINFRA_API_KEY in your environment.")

        client = OpenAI(
            api_key=api_key,
            base_url=DEEPINFRA_BASE_URL,
        )
        truncated = decision_text[:max_chars]

        resp = client.chat.completions.create(
            model=DEEPINFRA_GENERATION_MODEL,
            messages=[
                {"role": "system", "content": _CASE_CARD_SYSTEM},
                {
                    "role": "user",
                    "content": (
                        "Here is the tribunal decision:\n\n"
                        + truncated
                    ),
                },
            ],
            max_tokens=max_tokens,
            temperature=0.3,
        )
        return _sanitize_case_card_output(
            resp.choices[0].message.content,
        )


print("CaseCardGenerator defined.")

In [ ]:
# Generate a case card for the top-ranked result
generator = CaseCardGenerator()

top_decision_text = top_results[0]["text"]
print(f"Generating case card for: {top_results[0]['name']}\n")

case_card = generator.generate_case_card(
    top_decision_text,
    max_tokens=GENERATION_MAX_TOKENS,
    max_chars=GENERATION_MAX_CHARS,
)

print(case_card)

---
## 8. Full Pipeline

The pipeline orchestrator ties all three stages together into a single
`search()` call. It manages the data lifecycle (loading, indexing) and
dispatches to each stage sequentially.

The pipeline supports a **fast mode** that reduces the number of candidates
at each stage and shortens token budgets, trading a small amount of quality
for significantly faster response times.

| Parameter | Normal | Fast |
|-----------|--------|------|
| Stage 1 candidates | 40 | 20 |
| Stage 2 results | 5 | 3 |
| Reranker max tokens | 8,192 | 2,048 |
| Generation max tokens | 512 | 256 |
| Generation max chars | 24,000 | 12,000 |

In [ ]:
# ── Pipeline data structures ─────────────────────────────────────────

@dataclass
class SearchResult:
    """A single ranked decision returned to the user."""
    rank: int
    name: str
    date: str
    url: str
    reranker_score: float
    snippet: str
    full_text: str


@dataclass
class PipelineOutput:
    """Everything returned after a search."""
    case_card: str | None = None
    results: list[SearchResult] = field(default_factory=list)


# ── Pipeline orchestrator ────────────────────────────────────────────

class SSTNavigatorPipeline:
    """End-to-end two-stage RAG pipeline for SST decisions."""

    def __init__(self, dev_mode: bool = False, fast_mode: bool = False):
        self.dev_mode = dev_mode
        self.fast_mode = fast_mode
        self._df: pd.DataFrame | None = None
        self._searcher = SemanticSearcher()
        self._reranker = Reranker()
        self._generator = CaseCardGenerator()

    def _active_params(self) -> dict:
        """Return runtime parameters based on quality/speed mode."""
        if self.fast_mode:
            return {
                "stage1_top_k": FAST_STAGE1_TOP_K,
                "stage2_top_k": FAST_STAGE2_TOP_K,
                "reranker_max_tokens": FAST_RERANKER_MAX_TOKENS,
                "generation_max_tokens": FAST_GENERATION_MAX_TOKENS,
                "generation_max_chars": FAST_GENERATION_MAX_CHARS,
            }
        return {
            "stage1_top_k": STAGE1_TOP_K,
            "stage2_top_k": STAGE2_TOP_K,
            "reranker_max_tokens": RERANKER_MAX_TOKENS,
            "generation_max_tokens": GENERATION_MAX_TOKENS,
            "generation_max_chars": GENERATION_MAX_CHARS,
        }

    def load_data(self) -> int:
        """Load the SST dataset and return the row count."""
        max_rows = DEV_ROW_LIMIT if self.dev_mode else None
        self._df = load_sst_decisions(max_rows=max_rows)
        return len(self._df)

    def build_index(self) -> None:
        """Download cached embeddings and align with the DataFrame."""
        if self._df is None:
            raise RuntimeError("Call load_data() first.")

        if not self._searcher.load_embeddings_cache():
            raise RuntimeError("Could not load embedding cache.")

        target_urls = self._df["url_en"].astype(str).tolist()
        matched = self._searcher.align_to_urls(target_urls)

        if len(matched) < len(target_urls):
            self._df = self._df.iloc[matched].reset_index(drop=True)
            logger.warning(
                "Trimmed to %d rows with cached embeddings.",
                len(matched),
            )

    @property
    def is_ready(self) -> bool:
        return (
            self._df is not None and self._searcher.has_embeddings
        )

    def search(
        self,
        query: str,
        include_case_card: bool = True,
    ) -> PipelineOutput:
        """Run the full two-stage search + optional generation.

        1. Semantic search (bi-encoder)  → top candidates
        2. Rerank (cross-encoder)        → top results
        3. Generate case card for #1     → plain-language summary
        """
        if not self.is_ready:
            raise RuntimeError(
                "Pipeline not initialised. "
                "Call load_data() + build_index()."
            )

        params = self._active_params()

        # ── Stage 1: Semantic search ─────────────────────────────
        hits = self._searcher.search(
            query, top_k=params["stage1_top_k"],
        )
        logger.info("Stage 1 returned %d candidates.", len(hits))

        if not hits:
            return PipelineOutput(
                case_card="No similar cases found.",
                results=[],
            )

        # Build candidate dicts for the reranker
        candidates = []
        for idx, score in hits:
            row = self._df.iloc[idx]
            candidates.append({
                "index": idx,
                "name": row.get("name_en", "Unnamed"),
                "date": str(row.get("document_date_en", "")),
                "url": row.get("url_en", ""),
                "text": row["unofficial_text_en"],
                "stage1_score": score,
            })

        # ── Stage 2: Reranking ───────────────────────────────────
        top_results = self._reranker.rerank(
            query, candidates,
            top_k=params["stage2_top_k"],
            max_tokens=params["reranker_max_tokens"],
        )
        logger.info("Stage 2 returned %d results.", len(top_results))

        if not top_results:
            return PipelineOutput(
                case_card="No relevant cases after reranking.",
                results=[],
            )

        # ── Stage 3: Optional case-card generation ───────────────
        case_card: str | None = None
        if include_case_card:
            case_card = self._generator.generate_case_card(
                top_results[0]["text"],
                max_tokens=params["generation_max_tokens"],
                max_chars=params["generation_max_chars"],
            )

        # ── Assemble output ──────────────────────────────────────
        results = []
        for rank, r in enumerate(top_results, 1):
            results.append(SearchResult(
                rank=rank,
                name=r["name"],
                date=r["date"],
                url=r["url"],
                reranker_score=r["reranker_score"],
                snippet=r["text"][:SNIPPET_LENGTH],
                full_text=r["text"],
            ))

        return PipelineOutput(case_card=case_card, results=results)


print("SSTNavigatorPipeline defined.")

In [ ]:
# Instead of re-loading data and embeddings (already done above),
# we wire the existing searcher + DataFrame into a pipeline instance.

pipeline = SSTNavigatorPipeline(dev_mode=False, fast_mode=False)
pipeline._df = df
pipeline._searcher = searcher  # reuse the already-loaded embedder

print(f"Pipeline ready: {pipeline.is_ready}")
print(f"Decisions indexed: {len(df):,}\n")

# ── Run a full search ────────────────────────────────────────────────
query = (
    "I have chronic back pain and depression that prevent me "
    "from working. My CPP disability application was denied."
)
print(f"Query: {query!r}\n")

output = pipeline.search(query, include_case_card=True)

# ── Display the case card ────────────────────────────────────────────
if output.case_card:
    print("=" * 70)
    print("CASE CARD (Top Result Summary)")
    print("=" * 70)
    print(output.case_card)
    print()

# ── Display ranked results ───────────────────────────────────────────
print("=" * 70)
print("RANKED RESULTS")
print("=" * 70)
for r in output.results:
    print(f"\n#{r.rank}  {r.name}")
    print(f"    Date:  {r.date}")
    print(f"    Score: {r.reranker_score:.4f}")
    print(f"    URL:   {r.url}")
    print(f"    Snippet: {r.snippet[:200]}...")

### Additional Query Example

Let's run a second query to demonstrate the pipeline's versatility across
different benefit types and legal questions.

In [ ]:
query2 = (
    "I quit my job because my employer was harassing me and "
    "creating an unsafe work environment. Can I still get EI?"
)
print(f"Query: {query2!r}\n")

output2 = pipeline.search(query2, include_case_card=True)

if output2.case_card:
    print("=" * 70)
    print("CASE CARD")
    print("=" * 70)
    print(output2.case_card)
    print()

print("=" * 70)
print("RANKED RESULTS")
print("=" * 70)
for r in output2.results:
    print(f"\n#{r.rank}  {r.name}")
    print(f"    Date:  {r.date}")
    print(f"    Score: {r.reranker_score:.4f}")
    print(f"    URL:   {r.url}")

---
## 9. Conclusion

This notebook demonstrated a complete **two-stage RAG pipeline** for searching
Social Security Tribunal of Canada decisions using semantic AI:

1. **Data Loading** — 17,000+ SST decisions loaded from the A2AJ dataset on
   Hugging Face.

2. **Section-Aware Parsing** — Decisions are parsed into prioritised sections
   (Analysis, Issue, Conclusion, etc.) so that the most legally relevant
   content is always surfaced to the reranker, even for very long documents.

3. **Bi-Encoder Semantic Search** — Pre-computed embeddings enable sub-second
   retrieval of the top 40 candidates by cosine similarity, capturing semantic
   meaning rather than just keyword overlap.

4. **Cross-Encoder Reranking** — A more powerful model scores each candidate
   with full query-document attention, narrowing the field to the 5 most
   relevant decisions.

5. **Case Card Generation** — A plain-language summary makes the top result
   immediately understandable to self-represented litigants.

### Potential Future Work

- **Benefit-type filtering:** Pre-classify decisions by benefit area (EI, CPP,
  OAS) to allow scoped searches.
- **Citation graph:** Link decisions that cite each other to surface lines of
  authority.
- **Explainability:** Highlight which passages in a decision drove the reranker
  score.
- **Multilingual support:** Extend to French-language SST decisions.

---

> **Disclaimer:** This is an educational research tool and does not constitute
> legal advice. Decisions retrieved may not reflect the current state of the
> law. Please consult a licensed professional or your local legal-aid clinic
> for legal guidance.